# 📚 DDIA Bölüm 3: Data Models and Query Languages
## Veri Modelleri ve Sorgu Dilleri — Kapsamlı Türkçe Anlatım

> *"Dilin sınırları, dünyamın sınırlarıdır."*  
> — Ludwig Wittgenstein, Tractatus Logico-Philosophicus (1922)

---

Bu notebook, **Designing Data-Intensive Applications (DDIA)** kitabının 3. bölümünü,
tüm teknik terimler İngilizce korunarak Türkçe olarak açıklamaktadır.

**Kapsanan konular:**
- Data models neden bu kadar önemli?
- Declarative vs. Imperative query languages
- Relational Model vs. Document Model
- Object-Relational Mismatch & ORM
- One-to-many, many-to-one, many-to-many relationships
- Normalization & Denormalization & Joins
- Star Schema & Snowflake Schema (Analytics)
- Graph-Like Data Models
- Property Graphs & Cypher
- Triple Stores, RDF & SPARQL
- Datalog
- GraphQL
- Event Sourcing & CQRS
- DataFrames, Matrices ve Arrays

---
## 🧩 1. Data Models: Neden Bu Kadar Önemli?

**Data models** (veri modelleri), yazılım geliştirmenin belki de en kritik parçasıdır.
Nedeni şudur: Bir data model, yalnızca yazılımın nasıl yazıldığını değil, aynı zamanda
**çözmeye çalıştığımız problemi nasıl düşündüğümüzü** de derinden etkiler.

### Katmanlı abstraction (soyutlama) yapısı

Çoğu uygulama, birbirinin üstüne yığılmış data model katmanlarından oluşur.
Her katmanda temel soru şudur: *"Bir alt katman bu veriyi nasıl temsil eder?"*

**En yüksek seviyeden en düşük seviyeye katmanlar:**

| Katman | Açıklama |
|--------|----------|
| **Uygulama Geliştirici** | Gerçek dünyayı (insanlar, kuruluşlar, işlemler, sensörler vb.) **objects** veya **data structures** ve **APIs** cinsinden modeller |
| **General-purpose data model** | Bu yapılar **JSON / XML documents**, **relational tables** veya **graph** (vertices & edges) olarak ifade edilir |
| **Database storage engine** | Veritabanı, bu document/relational/graph veriyi **bytes** (bellekte, diskte veya ağda) cinsinden saklar. Bu veri **query, search, manipulate, process** edilebilir hale getirilir |
| **Hardware** | Mühendisler, byte'ları **elektrik akımı, ışık darbeleri, manyetik alanlar** vb. cinsinden temsil eder |

Karmaşık uygulamalarda bu katmanlar arasında ek **API'ler** bulunabilir. Ama temel fikir aynıdır:
> Her katman, altındaki katmanın karmaşıklığını **clean data model** sunarak gizler.

Bu **abstractions** (soyutlamalar), farklı grupların —örneğin veritabanı geliştiricileri ile uygulama geliştiricileri— birlikte etkin çalışabilmesini sağlar.

### Hangi data models bu bölümde inceleniyor?

- **Relational model** (SQL)
- **Document model** (JSON)
- **Graph-based data models**
- **Event sourcing**
- **DataFrames**

Her model için **trade-off**'lar (avantaj/dezavantaj dengesi) karşılaştırılacak.

---
## 🗣️ 2. Declarative Query Languages

### Declarative nedir?

Bu bölümde tartışılan sorgu dillerinin çoğu (**SQL, Cypher, SPARQL, Datalog**)
**declarative** (bildirimsel) dillerdir.

**Declarative bir dilde** şunu belirtirsiniz:
- Hangi **pattern**'deki veriyi istiyorsunuz
- Sonuçların hangi koşulları karşılaması gerektiği
- Verinin nasıl dönüştürüleceği (sorted, grouped, aggregated)
- **Ama bu hedefe nasıl ulaşılacağını belirtmezsiniz!**

**Query optimizer**, hangi **indexes** ve **join algorithms** kullanılacağına,
sorgunun hangi parçalarının hangi sırayla çalıştırılacağına kendisi karar verir.

### Imperative (Zorunlu) dil karşılaştırması

Python veya Java gibi çoğu programlama dilinde **imperative** (zorunlu) bir yaklaşım
kullanılır: Bilgisayara, hangi işlemleri hangi sırayla yapacağını adım adım anlatırsınız.

```
# Imperative yaklaşım (pseudocode)
sonuclar = []
for satir in tablo:
    if satir.kategori == 'Köpekbalığı':
        sonuclar.append(satir)
sonuclar.sort(key=lambda x: x.isim)
```

```sql
-- Declarative yaklaşım (SQL)
SELECT * FROM gozlemler
WHERE aile = 'Sharks'
ORDER BY isim;
```

### Declarative dilin avantajları

1. **Daha özlü ve okunabilir** — bir algoritmadan çok daha az kod
2. **Query engine'in implementation detaylarını gizler** — veritabanı sistemi,
   sizin query'lerinizi değiştirmenize gerek kalmadan **performance improvements**
   yapabilir
3. **Otomatik paralellik** — veritabanı, declarative sorguyu birden fazla CPU core
   ve makinede paralel olarak çalıştırabilir; siz bunu elle kodlamak zorunda kalmazsınız
4. **El yazısı (handcoded) algoritmada** bu tür paralel execution'ı kendiniz uygulamak
   büyük iş olurdu

---
## 🔄 3. Relational Model vs. Document Model

### Relational Model'in tarihi

Bugün en çok bilinen data model, **Edgar Codd**'un 1970'te önerdiği **relational model**
üzerine inşa edilmiş **SQL**'dir.

Bu modelde veri **relations** (SQL'de **tables**) içinde düzenlenir. Her **relation**,
sırasız bir **tuples** (SQL'de **rows**) koleksiyonudur.

**Relational model başlangıçta teorik bir öneriydi.** Zamanla 1980'lerin ortasına
gelindiğinde **RDBMS (Relational Database Management Systems)** ve **SQL**,
düzenli yapılı veri saklamak ve sorgulamak için standart araçlar haline geldi.

### Rakip yaklaşımların tarihi

| Dönem | Yaklaşım | Sonuç |
|-------|----------|-------|
| 1970'ler–1980'ler | **Network model** ve **Hierarchical model** | Relational model bunları geride bıraktı |
| 1980'lerin sonu – 1990'ların başı | **Object databases** | Geldi ve gitti |
| 2000'lerin başı | **XML databases** | Yalnızca niche kullanımla sınırlı kaldı |
| 2010'lar | **NoSQL** (buzzword) | Kalıcı etkiler bıraktı ama SQL'i ölduramadı |

**SQL** de zamanla **XML, JSON, graph data** desteği ekleyerek genişledi.

### NoSQL hareketi

**NoSQL**, tek bir teknolojiyi değil, şu fikirlerin etrafında gelişen gevşek bir kümeyi ifade eder:
- Yeni **data models**
- **Schema flexibility** (esnek şema)
- **Scalability** (ölçeklenebilirlik)
- **Open source** lisans modellerine yönelim

**NewSQL** ise **NoSQL** sistemlerinin ölçeklenebilirliğini, geleneksel ilişkisel veritabanlarının
data model'i ve **transactional guarantees**'ı ile birleştirmeyi amaçladı.

NoSQL'in kalıcı etkilerinden biri, **document model**'in yaygınlaşmasıdır. Veriyi genellikle
**JSON** olarak temsil eden bu model, başlangıçta **MongoDB** ve **Couchbase** gibi
özel **document databases** tarafından popüler hale getirildi. Günümüzde relational
veritabanlarının çoğu da **JSON** desteği eklemiştir.

---
## ⚡ 4. Object-Relational Mismatch ve ORM

### Impedance Mismatch nedir?

Günümüz uygulama geliştirmenin büyük bölümü **object-oriented programming languages**
ile yapılmaktadır. Bu da SQL data model'ine yönelik yaygın bir eleştiriye yol açar:
Eğer veri **relational tables** içinde saklanıyorsa, uygulama kodundaki **objects** ile
veritabanının **tables/rows/columns** modeli arasında beceriksiz bir çeviri katmanı gerekir.

Bu iki model arasındaki kopuşa **impedance mismatch** (empedans uyumsuzluğu) denir.

> 💡 **Elektronik bağlantısı:** Terim elektroniğe dayanır — iki devreyi birbirine bağladığınızda,
> çıkış ve giriş **impedances** eşleşirse güç transferi maksimuma ulaşır.
> Uyumsuzluk, sinyal yansımalarına ve diğer sorunlara yol açar.

### ORM (Object-Relational Mapping) nedir?

**ORM frameworks** (**ActiveRecord**, **Hibernate** gibi), bu çeviri katmanı için
gereken **boilerplate code** miktarını azaltır.

### ORM'nin dezavantajları

**1. İki modeli de düşünmek zorunda kalırsınız:**
ORM'ler iki model arasındaki farkları tamamen gizleyemez; geliştiriciler yine de
hem **relational** hem **object** temsillerini düşünmek zorunda kalır.

**2. Yalnızca OLTP için:**
ORM'ler genellikle yalnızca **OLTP (Online Transaction Processing)** uygulama
geliştirme için kullanılır. Veriyi analytics amacıyla hazırlayan veri mühendisleri,
altta yatan **relational representation** ile doğrudan çalışmak zorundadır.

**3. Farklı veri sistemleri desteği sınırlı:**
Search engine'ler, **graph databases** ve **NoSQL** sistemleri içeren kuruluşlar,
ORM desteğinin yetersiz kaldığını görebilir.

**4. Otomatik oluşturulan schema sorunları:**
Bazı ORM'ler otomatik **relational schema** üretir; ancak bunlar, veriye doğrudan
erişen kullanıcılar için garip olabilir ve veritabanında verimsiz çalışabilir.
ORM'nin schema ve query üretimini özelleştirmek karmaşık ve zahmetsizleştirme
avantajını ortadan kaldırabilir.

**5. N+1 Query Problem:**
ORM'ler, farkında olmadan verimsiz sorgular yazmayı kolaylaştırır.
Klasik örnek **N+1 query problem**'dir:

- N adet yorum döndüren bir sorgu yaparsınız, her yorum yazar ID'si içerir.
- Yazar adını göstermek için, her yorumun ID'si için `users` tablosuna ayrı sorgu atarsınız.
- Toplamda 1 (yorumlar) + N (yazarlar) = N+1 veritabanı sorgusu oluşur.
- El yazısı SQL'de bu **join** tek sorguda halledilebilirdi.
- Bunu önlemek için ORM'ye yorum çekerken yazar bilgisini de getirmesini söylemek gerekir.

### ORM'nin avantajları

1. **Boilerplate azaltır** — Relational ve in-memory object temsili arasındaki basit/tekrarlayan
   çeviriler için çok daha az kod yazılır. Karmaşık sorgular ORM dışında ele alınabilir.
2. **Query caching** — Bazı ORM'ler, veritabanı yükünü azaltmaya yardımcı olabilecek
   query sonuçlarını cache'ler.
3. **Schema migrations** — ORM'ler, **schema migrations** ve diğer yönetimsel
   işlemlerin yönetiminde yardımcı olabilir.

---
## 📄 5. One-to-Many Relationships ve Document Model

### LinkedIn profil örneği

Relational model'in bir sınırlamasını göstermek için LinkedIn profilini (bir özgeçmiş)
ele alalım. Profilin tamamı benzersiz bir `user_id` ile tanımlanabilir.
`first_name` ve `last_name` gibi alanlar kullanıcı başına tam olarak bir kez görünür;
bunlar `users` tablosundaki **columns** olarak modellenebilir.

Ama çoğu kişinin birden fazla işi (pozisyonu) vardır. Eğitim geçmişi değişen sayıda olabilir.
İletişim bilgileri de aynı şekilde değişken.

Bu **one-to-many relationships** (bire-çok ilişkileri) relational modelde nasıl temsil edilir?

### Relational yaklaşım: Ayrı tablolar

```sql
-- positions, education ve contact_info
-- her biri users tablosuna foreign-key reference ile
-- ayrı tablolarda tutulur
CREATE TABLE users (user_id INT PRIMARY KEY, first_name TEXT, ...);
CREATE TABLE positions (id INT, user_id INT REFERENCES users, job_title TEXT, ...);
CREATE TABLE education (id INT, user_id INT REFERENCES users, school TEXT, ...);
```

### Document (JSON) yaklaşımı: Daha doğal

```json
{
  "user_id": 251,
  "first_name": "Barack",
  "last_name": "Obama",
  "headline": "Former President of the United States",
  "positions": [
    {"job_title": "President", "organization": "United States of America"},
    {"job_title": "US Senator (D-IL)", "organization": "United States Senate"}
  ],
  "education": [
    {"school_name": "Harvard University", "start": 1988, "end": 1991},
    {"school_name": "Columbia University", "start": 1981, "end": 1983}
  ],
  "contact_info": {
    "website": "https://barackobama.com"
  }
}
```

### JSON'un avantajları

**1. Daha iyi locality:**
Relational örnekte profili çekmek için ya her tabloya ayrı sorgu atmak,
ya da `users` tablosu ile alt tablolar arasında karmaşık bir **multiway join** yapmak gerekir.
JSON temsilinde ilgili tüm bilgi tek yerde bulunur; bu hem sorguyu daha hızlı hem daha basit yapar.

**2. Tree structure (ağaç yapısı) görünür:**
Kullanıcı profili ile pozisyonlar/eğitim/iletişim arasındaki **one-to-many relationships**,
verideki bir ağaç yapısını ima eder; JSON temsili bu ağaç yapısını açıkça ortaya koyar.

> 📝 **One-to-few notu:** Bir **one-to-many** ilişkisi bazen **one-to-few** olarak da adlandırılır;
> çünkü bir özgeçmişte genellikle az sayıda pozisyon bulunur.
> Ancak gerçekten çok sayıda ilişkili öğe varsa —mesela bir ünlünün sosyal medya gönderisindeki
> binlerce yorum— bunların tamamını aynı document içine gömmek pratik olmayabilir;
> bu durumda Figure 3-1'deki relational yaklaşım tercih edilebilir.

---
## 🔗 6. Normalization, Denormalization ve Joins

### Neden ID kullanılır? Normalization nedir?

LinkedIn profil örneğinde `region_id`, düz metin olan `"Washington, DC"` yerine bir ID
olarak saklanmaktadır. Neden?

**Standart coğrafi bölge listesi ve açılır menü kullanmanın avantajları:**

- **Tutarlı yazım ve stil** — profiller arasında aynı yazım
- **Belirsizlik yok** — aynı isme sahip birden fazla yer varsa karışıklık olmaz
  (örn. Washington, DC mi yoksa Washington eyaleti mi?)
- **Kolay güncelleme** — isim yalnızca bir yerde saklanır; değişmesi gerekirse
  tek yerden güncellenir (örn. siyasi nedenle şehir adı değiştiğinde)
- **Lokalizasyon desteği** — site başka dillere çevrildiğinde, standardize listeler
  yerelleştirilebilir; bölge, görüntüleyicinin dilinde gösterilebilir
- **Daha iyi arama** — "ABD Doğu Kıyısındaki" kişileri aradiğınızda, listeler
  Washington'ın Doğu Kıyısında olduğunu kodlayabilir; bu bilgi tek başına
  `"Washington, DC"` dizgisinden çıkarılamaz

### ID vs. Plain Text: Normalization

**ID kullanırsanız** → Veri daha **normalized** (normalleştirilmiş) olur:
İnsanlara anlamlı olan bilgi (`"Washington, DC"`) yalnızca bir yerde saklanır.
Her referans, yalnızca veritabanı içinde anlamlı olan bir **ID** kullanır.

**Düz metin saklarsanız** → **Denormalized** (normalleştirilmemiş) olur:
İnsanlara anlamlı bilgi, onu kullanan her kayıtta kopyalanır.

**ID kullanmanın avantajı:**
Bir ID'nin insanlar için hiçbir anlamı olmadığından, değişmesi gerekmez;
tanımladığı bilgi değişse bile ID aynı kalabilir. İnsanlar için anlamlı olan her
şey gelecekte değişebilir — ve o bilgi kopyalanmışsa, tüm yedek kopyaların
güncellenmesi gerekir. Bu durum daha fazla kod, daha fazla yazma işlemi, daha
fazla disk alanı gerektirir ve tutarsızlık riskini doğurur.

### Joins neden gereklidir?

**Normalized representation**'ın dezavantajı: Bir ID içeren kaydı görüntülemek
istediğinizde, ID'yi insan tarafından okunabilir bir şeye çözümlemek için
ek bir **lookup (arama)** yapmanız gerekir. Relational modelde bu işlem **join** ile yapılır:

```sql
SELECT users.*, regions.region_name
FROM users
JOIN regions ON users.region_id = regions.id
WHERE users.id = 251;
```

### Document databases'de Joins

**Document databases** hem **normalized** hem **denormalized** veriyi saklayabilir;
ancak çoğunlukla **denormalization** ile ilişkilendirilir. Bunun iki nedeni vardır:

1. **JSON data model**, ek **denormalized fields** saklamayı kolaylaştırır.
2. Birçok document database'de **join desteği** zayıftır; bu da
   **normalization**'ı elverişsiz kılar.

Bazı document database'ler **join**'i hiç desteklemez; bu durumda **application code**
içinde join yapmanız gerekir: Önce ID içeren bir document çekersiniz, sonra o
ID'yi çözümlemek için ikinci bir sorgu atarsınız.

**MongoDB**'de `$lookup` operatörü ile bir **aggregation pipeline** içinde join yapabilirsiniz:

```javascript
db.users.aggregate([
  { $match: { _id: 251 } },
  { $lookup: {
      from: "regions",
      localField: "region_id",
      foreignField: "_id",
      as: "region"
  } }
])
```

---
## ⚖️ 7. Normalization Trade-off'ları

### Özgeçmiş örneğinde detay

Özgeçmiş örneğinde `region_id` alanı standart bölge listesine bir referans iken,
`organization` (çalışılan şirket veya devlet kurumu) ve `school_name` (öğrenim görülen
kurum) birer **string**'dir. Bu temsil **denormalized**'dır: Aynı şirkette çalışan
pek çok kişi olabilir ama bunları birbirine bağlayan bir ID yoktur.

Şirket/okul isimlerinin de **entity** (varlık) yapılıp ID ile referans edilmesi
değerlendirilebilir. Örneğin, ismin yanı sıra **logo URL**'si de göstermek isteseydik:

**Denormalized yaklaşım:**
Her bireysel profilin JSON'una logo'nun **image URL**'sini dahil ederdiniz.
Bu JSON document'ı bağımsız kılar; ama ileride logo değişirse, eski URL'nin
tüm geçtiği yerler bulunup güncellenmelidir.

**Normalized yaklaşım:**
Bir kuruluşu veya okulu temsil eden bir **entity** oluşturulur; adı, logo URL'si ve
belki başka öznitelikleri (açıklama, haber akışı vb.) bu entity'nin bir parçası olarak
bir kez saklanır. Bunu referans eden her özgeçmiş yalnızca ID'ye başvurur;
logo güncellemesi tek yerden yapılır.

### Genel kural

| Yaklaşım | Yazma | Okuma |
|----------|-------|-------|
| **Normalized** | Daha hızlı (tek kopya) | Daha yavaş (join gerektirir) |
| **Denormalized** | Daha yavaş (birden fazla kopya güncellenmeli) | Daha hızlı (join azalır) |

**Denormalization**'ı **derived data** (türetilmiş veri) olarak düşünmek faydalıdır:
Yedek kopyaları güncel tutmak için bir süreç kurmanız gerekir.

Tüm bu güncellemelerin maliyetinin yanı sıra, **crash recovery** sırasında tutarlılık
sorununu da göz önünde bulundurmanız gerekir. **Atomic transactions** sunan veritabanları
tutarlılığı korumayı kolaylaştırır; ancak tüm veritabanları birden fazla document
genelinde **atomicity** sunmaz.

### OLTP vs. Analytics için normalization

- **OLTP sistemleri** için normalization genellikle daha iyidir: Hem okuma hem güncellemeler
  hızlı olmalıdır.
- **Analitik sistemler** çoğunlukla **denormalized** veri ile daha iyi çalışır:
  Güncellemeler toplu yapılır ve **read-only** sorgu performansı baskın endişedir.
- **Küçük–orta ölçekli sistemlerde** normalized data model çoğunlukla en iyisidir:
  Birden fazla veri kopyasının tutarlılığını sağlamak zorunda kalmazsınız ve
  join maliyeti kabul edilebilir düzeydedir.
- **Çok büyük ölçekli sistemlerde** join maliyeti sorunlu hale gelebilir.

### Sosyal ağ case study'sinde denormalization

Bölüm 2'de incelenen sosyal ağ örneğinde, **normalized representation** ile
**precomputed, materialized timelines** (denormalized) karşılaştırılmıştı.
Gönderiler ile takip işlemleri arasındaki **join** çok pahalıydı ve
**materialized timeline**, o join'in sonucunun bir **cache**'idir.

X'in (eski adıyla Twitter) **materialized timeline** implementasyonu her gönderinin
metnini değil, yalnızca **post ID**, **sender_id** ve birkaç ek bilgiyi saklar.
Timeline okunurken servis hâlâ iki **join** yapar: Gerçek gönderi içeriği ve
gönderenin profili ID'leri aracılığıyla aranır.
ID'leri insan tarafından okunabilir bilgiye dönüştürme sürecine **hydrating the IDs** denir
ve bu esasen **application code** içinde yapılan bir join'dir.

Bu örnekten çıkan kritik sonuç: **Veri okurken join yapmak**, bazen iddia edildiği gibi,
yüksek performanslı ve ölçeklenebilir hizmetler oluşturmanın önünde bir engel değildir.
**Hydrating** post ve user ID'leri oldukça paralel işlenebilir ve maliyet, takip ettiğiniz
veya takip eden hesap sayısına bağlı değildir.

**Normalization ve Denormalization** özünde iyi ya da kötü değildir — yalnızca
okuma/yazma performansı ve implementasyon çabası açısından **trade-off**'ları temsil eder.
Ne kadar sık değiştiği ve okuma/yazma maliyetini dikkatlice değerlendirmek gerekir.

---
## 🔀 8. Many-to-One ve Many-to-Many Relationships

### İlişki türleri

- **One-to-many (bire-çok):** Bir özgeçmişin birçok pozisyonu vardır; ama her pozisyon yalnızca
  bir özgeçmişe aittir. Örnek: `positions` ve `education` tabloları.
- **Many-to-one (çoka-bir):** Birçok kişi aynı bölgede yaşayabilir; ama her kişi
  aynı anda yalnızca bir bölgede yaşar. Örnek: `region_id` alanı.
- **Many-to-many (çoka-çok):** Bir kişi birden fazla kuruluşta çalışmış olabilir;
  bir kuruluşun geçmiş veya mevcut birçok çalışanı vardır.

### Relational modelde many-to-many

Relational modelde **many-to-many** ilişkisi genellikle bir **associative table** veya
**join table** ile temsil edilir. Aşağıdaki örnekte her `positions` satırı, bir
`user_id` ile bir `org_id`'yi ilişkilendirir:

```sql
CREATE TABLE positions (
  id     INT PRIMARY KEY,
  user_id INT REFERENCES users(id),
  org_id  INT REFERENCES organizations(id),
  job_title TEXT,
  start_year INT,
  end_year   INT
);
```

### Document modelde many-to-many

**Many-to-one** ve **many-to-many** ilişkileri, tek başına bağımsız (self-contained)
bir JSON document içine pek sığmaz; bunlar normalized temsile daha yatkındır.

```json
{
  "user_id": 251,
  "positions": [
    {"start": 2009, "end": 2017, "job_title": "President",       "org_id": 513},
    {"start": 2005, "end": 2008, "job_title": "US Senator (D-IL)", "org_id": 514}
  ]
}
```

**Many-to-many** ilişkiler çoğunlukla her iki yönde sorgulanmak istenir:
- Belirli bir kişinin çalıştığı tüm kuruluşlar
- Belirli bir kuruluşta çalışmış tüm kişiler

Bu tür sorguları etkinleştirmenin bir yolu, ID referanslarını her iki tarafta da saklamaktır.
Ancak bu **denormalized** bir temsil olur; ilişki iki yerde saklanır ve tutarsız hale gelebilir.

**Normalized temsil** ilişkiyi yalnızca bir yerde saklar ve her iki yönde sorgulanabilmesi
için **secondary indexes**'e güvenir.

---
## 📊 9. Stars and Snowflakes: Analytics için Schemas

### Data Warehouse schema'ları

**Data warehouses** genellikle **relational**'dır ve tablolar için birkaç yaygın
convention kullanılır:

- **Star schema**
- **Snowflake schema**
- **Dimensional modeling**
- **One Big Table (OBT)**

Bu yapılar **business analysts**'ların ihtiyaçları için optimize edilmiştir.
**ETL (Extract, Transform, Load)** süreçleri, operasyonel sistemlerden veriyi
seçilen schema'ya dönüştürür.

### Star Schema

Bir perakende zincirinin data warehouse'unda yer alabilecek **star schema** örneğini
ele alalım. Schema'nın merkezinde bir **fact table** bulunur (bu örnekte `fact_sales`).

**Fact table'ın her satırı**, belirli bir zamanda gerçekleşen bir **event**'i temsil eder:
burada her satır, bir müşterinin bir ürün satın almasını temsil eder.
Web trafiğini analiz etseydiniz, her satır bir sayfa görüntülemesini veya tıklamayı temsil edebilirdi.

**Olaylar genellikle bireysel events olarak yakalanır** — bu, daha sonra maksimum
analiz esnekliği sağlar. Ancak bu da **fact table**'ın son derece büyük olabileceği anlamına gelir.
Büyük bir kurumun data warehouse'u çoğunlukla **petabytes** boyutunda işlem geçmişi içerir.

### Fact table ve Dimension tables

**Fact table**'daki bazı sütunlar **attributes**'tir (ürünün satış fiyatı, tedarikçiden
alış fiyatı gibi — kâr marjı hesaplamaya imkân tanır).

Diğer sütunlar, **dimension tables** adı verilen diğer tablolara **foreign-key** referanslarıdır.
Her **fact table** satırı bir event'i temsil ettiğinden, **dimensions** olayın
*kim, ne, nerede, ne zaman, nasıl ve neden*'ini temsil eder.

Örneğin:
- `dim_product` → satılan ürünün bilgileri (SKU, açıklama, marka, kategori, yağ içeriği, paket boyutu)
- `dim_date` → tarih ve saat bilgisi (tatil günleri gibi ek bilgiler de içerebilir)
- `dim_store` → mağaza bilgileri (sunulan hizmetler, metrekare, açılış tarihi vb.)

### Star schema'nın adı

Tablo ilişkileri görselleştirildiğinde **fact table** ortadadır; **dimension tables**
onu çevreler (bir yıldızın ışınları gibi).

### Snowflake Schema

Bu şablonun bir varyasyonu olan **snowflake schema**'da **dimensions** daha da
**subdimensions**'a bölünür. Örneğin marka ve ürün kategorileri için ayrı tablolar
olabilir; `dim_product` tablosundaki her satır bu değerleri string olarak saklamak
yerine marka ve kategoriye **foreign key** ile referans verir.

**Snowflake schemas**, star schemas'tan daha **normalized**'dır; ancak star schemas,
analistlerin çalışması için daha basit olduğundan genellikle tercih edilir.

### Tipik data warehouse tablo genişliği

Tipik bir data warehouse'da tablolar çoğunlukla oldukça geniştir:
- **Fact tables** sık sık 100'den fazla, bazen birkaç yüz sütun içerir.
- **Dimension tables** da geniş olabilir: `dim_store` tablosu sunulan hizmetler,
  içeride fırın olup olmadığı, metrekare, açılış tarihi, en yakın otoyola uzaklık
  gibi pek çok bilgi içerebilir.

### One Big Table (OBT)

Bazı data warehouse schema'ları **denormalization**'ı daha da ileri götürür ve
**dimension tables**'ı tamamen kaldırır; dimension bilgilerini denormalized sütunlar
olarak **fact table**'a katar (esasen fact table ile dimension tables arasındaki
join'i önceden hesaplar). Bu yaklaşım **OBT (One Big Table)** olarak bilinir.
Daha fazla depolama alanı gerektirse de bazen daha hızlı sorgular sağlar.

Analytics bağlamında bu tür **denormalization** sorunsuz kabul edilir:
Veri genellikle değişmeyecek tarihsel olayların bir günlüğünü temsil eder.
OLTP sistemlerindeki **data consistency** ve yazma yükü sorunları, analytics'te o kadar
acil değildir.

---
## 🤔 10. Hangi Model Ne Zaman Kullanılır?

### Document model lehine argümanlar

1. **Schema flexibility** (şema esnekliği)
2. **Locality** sayesinde daha iyi performans
3. Bazı uygulamalar için uygulama object modeline daha yakın olması

### Relational model lehine argümanlar

1. **Join** desteği için daha iyi
2. **Many-to-one** ve **many-to-many** ilişkiler için daha iyi

### Ne zaman Document model tercih edilmeli?

Uygulamanızdaki verinin **document-like** bir yapısı varsa
(yani genellikle tamamı birden yüklenen **one-to-many** ilişkilerden oluşan bir ağaç),
**document model** kullanmak muhtemelen iyi bir fikirdir.

Relational modelin **shredding** tekniği — document-like bir yapıyı birden fazla tabloya bölmek
(positions, education, contact_info gibi) — hantal schema'lara ve gereksiz karmaşık
uygulama koduna yol açabilir.

### Document model'in sınırlamaları

**Document model**'in sınırlamaları da vardır. Örneğin, bir document içindeki
iç içe (nested) bir öğeye doğrudan referans veremezsiniz; bunun yerine
"251 numaralı kullanıcının pozisyon listesindeki ikinci öğe" gibi bir şey söylemeniz gerekir.
**Join**'lere ihtiyaç duyulduğunda, herhangi bir öğeye ID ile doğrudan referans
verebildiğinizden relational yaklaşım daha iyi çalışır.

**Sıralanabilir liste gereksinimleri:**
Bazı uygulamalar kullanıcının öğe sırasını belirlemesine izin verir (örneğin görevleri
sürükleyip bırakarak yeniden sıralar). **Document model** bu tür uygulamaları iyi destekler:
öğeler (veya ID'leri) sırayı belirlemek için bir JSON dizisinde saklanabilir.
Relational veritabanlarında yeniden sıralanabilir listeler için standart bir yol yoktur;
tam sayı sütununa göre sıralama (ortaya ekleme sırasında yeniden numaralandırma gerektirir),
ID'lerin bağlı liste tutulması veya **fractional indexing** gibi çeşitli çözümler kullanılır.

### Schema flexibility: Schema-on-Read vs. Schema-on-Write

Çoğu **document database** ve relational database'lerdeki **JSON desteği**,
document içindeki veriye herhangi bir **schema** uygulamaz.

Document databases bazen **schemaless** (şemasız) olarak adlandırılır, ama bu yanıltıcıdır:
Veriyi okuyan kod genellikle belirli bir yapı olduğunu varsayar — yani **implicit schema**
(örtük şema) vardır, ama veritabanı bunu zorunlu kılmaz.

Daha doğru terim:
- **Schema-on-read** — Verinin yapısı örtüktür; yalnızca veri okunduğunda yorumlanır.
  (Dynamic / runtime type checking'e benzer)
- **Schema-on-write** — Geleneksel relational database yaklaşımı: Schema açıktır ve
  veritabanı, veri yazılırken tüm verinin bu schema'ya uymasını sağlar.
  (Static / compile-time type checking'e benzer)

**Schema-on-read ne zaman avantajlı?**
- Koleksiyondaki öğelerin hepsi aynı yapıya sahip değilse (heterojen veri)
- Obje türü çok fazla ve her türü kendi tablosuna koymak pratik değilse
- Verinin yapısı, kontrol etmediğiniz ve her an değişebilecek dış sistemler
  tarafından belirleniyorsa

**Schema-on-write ne zaman avantajlı?**
Tüm kayıtların aynı yapıya sahip olması bekleniyorsa, schema'lar bu yapıyı
belgelemek ve uygulamak için kullanışlı bir mekanizmadır.

### Data locality for reads and writes

Bir document genellikle tek bir sürekli dizi olarak saklanır (**JSON, XML** veya
**MongoDB'nin BSON**'u gibi binary varyant). Uygulamanızın genellikle tüm
document'a erişmesi gerekiyorsa (örneğin bir web sayfasında render etmek için),
bu **storage locality** bir performans avantajı sağlar.

Veri birden fazla tabloya bölünmüşse, tüm veriyi çekmek için birden fazla
**index lookup** gerekir; bu daha fazla **disk seek** ve daha uzun süre anlamına gelir.

**Locality avantajı kısıtları:**
- Yalnızca büyük bir document'ın büyük bölümlerine aynı anda ihtiyaç duyulduğunda geçerlidir.
- Veritabanı genellikle tüm document'ı yüklemek zorundadır; büyük bir document'ın
  küçük bir bölümüne erişmek istiyorsanız bu israftır.
- Bir document güncellendiğinde tüm document yeniden yazılmalıdır. Bu nedenle
  document'ları küçük tutmak ve sık yapılan küçük güncellemelerden kaçınmak önerilir.

**Locality yalnızca document model'e özgü değildir:**
Google'ın **Spanner** veritabanı, relational modelde aynı locality özelliklerini sunar;
schema'nın bir tablonun satırlarının başka bir tablo içinde **interleaved (iç içe)** olarak
bulunacağını bildirmesine izin verir.
**Oracle** da benzer bir özellik sunar: **multi-table index cluster tables**.
Google'ın **Bigtable**'ı tarafından popüler hale getirilen ve **HBase** ile **Accumulo**'da
da kullanılan **wide-column data model**, **locality** yönetimi amacıyla **column families**'e sahiptir.

### Document ve relational databases'in yakınsaması

**Document databases** ve **relational databases** başlangıçta çok farklı yaklaşımlar
olarak ortaya çıktı; ama zamanla daha fazla benzer hale geldiler.

- **Relational databases**, **JSON types** ve **query operators** ile document içindeki
  property'leri index'leme yeteneği ekledi.
- Bazı **document databases** (**MongoDB, Couchbase, RethinkDB**), **join**'ler,
  **secondary indexes** ve **declarative query languages** için destek ekledi.

Bu yakınsama, uygulama geliştiriciler için iyi bir haberdir:
En iyi sonuç genellikle her ikisini aynı veritabanında birleştirebildiğinizde ortaya çıkar.

> 📝 **Tarihsel not:** Codd'un relational model açıklaması, bir satırdaki değerin ilkel
> bir sayı veya dize olmayabileceğini, iç içe bir ilişki (tablo) da olabileceğini
> ve böylece rasgele derinlikte iç içe geçmiş ağaç yapısının değer olarak kullanılabileceğini
> öngörüyordu. Buna **nonsimple domains** adını verdi. Bu, 30 yıldan fazla sonra
> SQL'e eklenen **JSON** ve **XML** desteğiyle karşılaştırılabilir.

---
## 🤔 10. Hangi Model Ne Zaman Kullanılır?

### Document model lehine argümanlar

1. **Schema flexibility** (şema esnekliği)
2. **Locality** sayesinde daha iyi performans
3. Bazı uygulamalar için uygulama object modeline daha yakın olması

### Relational model lehine argümanlar

1. **Join** desteği için daha iyi
2. **Many-to-one** ve **many-to-many** ilişkiler için daha iyi

### Ne zaman Document model tercih edilmeli?

Uygulamanızdaki verinin **document-like** bir yapısı varsa
(yani genellikle tamamı birden yüklenen **one-to-many** ilişkilerden oluşan bir ağaç),
**document model** kullanmak muhtemelen iyi bir fikirdir.

Relational modelin **shredding** tekniği — document-like bir yapıyı birden fazla tabloya bölmek
(positions, education, contact_info gibi) — hantal schema'lara ve gereksiz karmaşık
uygulama koduna yol açabilir.

### Document model'in sınırlamaları

**Document model**'in sınırlamaları da vardır. Örneğin, bir document içindeki
iç içe (nested) bir öğeye doğrudan referans veremezsiniz; bunun yerine
"251 numaralı kullanıcının pozisyon listesindeki ikinci öğe" gibi bir şey söylemeniz gerekir.
**Join**'lere ihtiyaç duyulduğunda, herhangi bir öğeye ID ile doğrudan referans
verebildiğinizden relational yaklaşım daha iyi çalışır.

**Sıralanabilir liste gereksinimleri:**
Bazı uygulamalar kullanıcının öğe sırasını belirlemesine izin verir (örneğin görevleri
sürükleyip bırakarak yeniden sıralar). **Document model** bu tür uygulamaları iyi destekler:
öğeler (veya ID'leri) sırayı belirlemek için bir JSON dizisinde saklanabilir.
Relational veritabanlarında yeniden sıralanabilir listeler için standart bir yol yoktur;
tam sayı sütununa göre sıralama (ortaya ekleme sırasında yeniden numaralandırma gerektirir),
ID'lerin bağlı liste tutulması veya **fractional indexing** gibi çeşitli çözümler kullanılır.

### Schema flexibility: Schema-on-Read vs. Schema-on-Write

Çoğu **document database** ve relational database'lerdeki **JSON desteği**,
document içindeki veriye herhangi bir **schema** uygulamaz.

Document databases bazen **schemaless** (şemasız) olarak adlandırılır, ama bu yanıltıcıdır:
Veriyi okuyan kod genellikle belirli bir yapı olduğunu varsayar — yani **implicit schema**
(örtük şema) vardır, ama veritabanı bunu zorunlu kılmaz.

Daha doğru terim:
- **Schema-on-read** — Verinin yapısı örtüktür; yalnızca veri okunduğunda yorumlanır.
  (Dynamic / runtime type checking'e benzer)
- **Schema-on-write** — Geleneksel relational database yaklaşımı: Schema açıktır ve
  veritabanı, veri yazılırken tüm verinin bu schema'ya uymasını sağlar.
  (Static / compile-time type checking'e benzer)

**Schema-on-read ne zaman avantajlı?**
- Koleksiyondaki öğelerin hepsi aynı yapıya sahip değilse (heterojen veri)
- Obje türü çok fazla ve her türü kendi tablosuna koymak pratik değilse
- Verinin yapısı, kontrol etmediğiniz ve her an değişebilecek dış sistemler
  tarafından belirleniyorsa

**Schema-on-write ne zaman avantajlı?**
Tüm kayıtların aynı yapıya sahip olması bekleniyorsa, schema'lar bu yapıyı
belgelemek ve uygulamak için kullanışlı bir mekanizmadır.

### Data locality for reads and writes

Bir document genellikle tek bir sürekli dizi olarak saklanır (**JSON, XML** veya
**MongoDB'nin BSON**'u gibi binary varyant). Uygulamanızın genellikle tüm
document'a erişmesi gerekiyorsa (örneğin bir web sayfasında render etmek için),
bu **storage locality** bir performans avantajı sağlar.

Veri birden fazla tabloya bölünmüşse, tüm veriyi çekmek için birden fazla
**index lookup** gerekir; bu daha fazla **disk seek** ve daha uzun süre anlamına gelir.

**Locality avantajı kısıtları:**
- Yalnızca büyük bir document'ın büyük bölümlerine aynı anda ihtiyaç duyulduğunda geçerlidir.
- Veritabanı genellikle tüm document'ı yüklemek zorundadır; büyük bir document'ın
  küçük bir bölümüne erişmek istiyorsanız bu israftır.
- Bir document güncellendiğinde tüm document yeniden yazılmalıdır. Bu nedenle
  document'ları küçük tutmak ve sık yapılan küçük güncellemelerden kaçınmak önerilir.

**Locality yalnızca document model'e özgü değildir:**
Google'ın **Spanner** veritabanı, relational modelde aynı locality özelliklerini sunar;
schema'nın bir tablonun satırlarının başka bir tablo içinde **interleaved (iç içe)** olarak
bulunacağını bildirmesine izin verir.
**Oracle** da benzer bir özellik sunar: **multi-table index cluster tables**.
Google'ın **Bigtable**'ı tarafından popüler hale getirilen ve **HBase** ile **Accumulo**'da
da kullanılan **wide-column data model**, **locality** yönetimi amacıyla **column families**'e sahiptir.

### Document ve relational databases'in yakınsaması

**Document databases** ve **relational databases** başlangıçta çok farklı yaklaşımlar
olarak ortaya çıktı; ama zamanla daha fazla benzer hale geldiler.

- **Relational databases**, **JSON types** ve **query operators** ile document içindeki
  property'leri index'leme yeteneği ekledi.
- Bazı **document databases** (**MongoDB, Couchbase, RethinkDB**), **join**'ler,
  **secondary indexes** ve **declarative query languages** için destek ekledi.

Bu yakınsama, uygulama geliştiriciler için iyi bir haberdir:
En iyi sonuç genellikle her ikisini aynı veritabanında birleştirebildiğinizde ortaya çıkar.

> 📝 **Tarihsel not:** Codd'un relational model açıklaması, bir satırdaki değerin ilkel
> bir sayı veya dize olmayabileceğini, iç içe bir ilişki (tablo) da olabileceğini
> ve böylece rasgele derinlikte iç içe geçmiş ağaç yapısının değer olarak kullanılabileceğini
> öngörüyordu. Buna **nonsimple domains** adını verdi. Bu, 30 yıldan fazla sonra
> SQL'e eklenen **JSON** ve **XML** desteğiyle karşılaştırılabilir.

---
## 🕸️ 11. Graph-Like Data Models

### Ne zaman graph model?

**Many-to-many** ilişkilerin çok yaygın olduğu durumlarda ne olur?

**Relational model** basit **many-to-many** ilişki durumlarını işleyebilir;
ama verideki bağlantılar giderek daha karmaşık hale geldikçe, veriyi
bir **graph** (çizge) olarak modellemek daha doğal hale gelir.

### Graph nedir?

Bir **graph**, iki tür nesneden oluşur:
- **Vertices** (aynı zamanda **nodes** veya **entities** olarak bilinir)
- **Edges** (aynı zamanda **relationships** veya **arcs** olarak bilinir)

### Tipik graph örnekleri

| Graph Türü | Vertices (Düğümler) | Edges (Kenarlar) |
|------------|---------------------|------------------|
| **Social graph** | İnsanlar | Kim kimi tanıyor |
| **Web graph** | Web sayfaları | HTML linkleri |
| **Yol/demiryolu ağı** | Kavşaklar | Yollar/demiryolları |

Bu graph'lar üzerinde iyi bilinen algoritmalar çalıştırılabilir:
- Harita navigasyon uygulamaları bir yol ağında iki nokta arasındaki en kısa yolu bulur.
- **PageRank**, web graph üzerinde bir web sayfasının popülerliğini (dolayısıyla arama
  sırasını) belirlemek için kullanılabilir.

### Graph temsil yöntemleri

1. **Adjacency list (komşuluk listesi):** Her **vertex**, bir kenar uzaklığındaki
   komşu **vertex**'lerin ID'lerini saklar.
   → Graph traversal'lar (dolaşımları) için iyidir.

2. **Adjacency matrix (komşuluk matrisi):** İki boyutlu bir dizi; her satır ve
   sütun bir **vertex**'e karşılık gelir. İki **vertex** arasında kenar yoksa 0, varsa 1.
   → **Machine learning** için uygundur.

### Heterojen graph'lar

Graph'lar homojen veriye sınırlı değildir. Graph'ların eşit derecede güçlü bir
kullanımı, tek bir veritabanında tamamen farklı türde nesneleri tutarlı biçimde
saklamaktır:

- **Facebook**, birçok **vertex** ve **edge** türünden oluşan tek bir graph tutar.
  Vertex'ler: kişiler, konumlar, olaylar, check-in'ler ve kullanıcı yorumları.
  Edge'ler: arkadaşlık, check-in konumu, yorumlara katkı, etkinliklere katılım.
  
- **Search engines**, kuruluşlar, kişiler ve mekanlar gibi sık aranan entity'ler
  hakkında bilgileri kaydetmek için **knowledge graphs** kullanır.
  **Wikidata** gibi bazı siteler, graph verisini yapılandırılmış biçimde yayımlar.

### Graph query language'leri

Bu bölümde incelenecek graph temsili ve sorgu dilleri:

| Model | Implementasyonlar |
|-------|------------------|
| **Property graph model** | Neo4j, Memgraph, KùzuDB |
| **Triple store model** | Datomic, AllegroGraph, Blazegraph |
| Amazon Neptune | Her ikisini de destekler |

Sorgu dilleri:
- **Cypher** (openCypher standardı)
- **SPARQL**
- **Datalog**
- **GraphQL**
- **SQL** (WITH RECURSIVE ile)
- Diğer: Gremlin, GSQL (TigerGraph), PGQL, **GQL** (ISO standardı, 2024)

---
## 🏷️ 12. Property Graphs

### Property graph model nedir?

**Property graph** (labeled property graph olarak da bilinir) modelinde her **vertex** şunları içerir:

- Benzersiz bir **identifier** (tanımlayıcı)
- Bu vertex'in temsil ettiği nesne türünü açıklayan bir **label** (string)
- Giden **edges** kümesi (**outgoing edges**)
- Gelen **edges** kümesi (**incoming edges**)
- **Properties** koleksiyonu (key-value çiftleri)

Her **edge** şunları içerir:

- Benzersiz bir **identifier**
- Edge'in başladığı **vertex** (**tail vertex**)
- Edge'in bittiği **vertex** (**head vertex**)
- İki **vertex** arasındaki ilişki türünü açıklayan bir **label**
- **Properties** koleksiyonu (key-value çiftleri)

### Property graph'ı relational schema ile temsil etmek

Bir **graph store**'u, biri **vertex**'ler diğeri **edge**'ler için olmak üzere iki
**relational table**'dan oluşuyor olarak düşünebilirsiniz:

```sql
CREATE TABLE vertices (
    vertex_id   integer PRIMARY KEY,
    label       text,
    properties  jsonb          -- key-value properties
);

CREATE TABLE edges (
    edge_id     integer PRIMARY KEY,
    tail_vertex integer REFERENCES vertices (vertex_id),  -- başlangıç noktası
    head_vertex integer REFERENCES vertices (vertex_id),  -- bitiş noktası
    label       text,
    properties  jsonb
);

-- Her iki yönde de traverse edebilmek için index'ler
CREATE INDEX edges_tails ON edges (tail_vertex);
CREATE INDEX edges_heads ON edges (head_vertex);
```

### Model'in önemli özellikleri

1. **Herhangi bir vertex diğeriyle edge ile bağlanabilir** — Hangi şeylerin birbiriyle
   ilişkilendirilebileceğini kısıtlayan bir schema yoktur.

2. **Hem gelen hem giden edge'leri verimli bulabilirsiniz** — Bu nedenle hem ileri
   hem geri yönde graph'ı **traverse** edebilirsiniz. (Bu yüzden örnek schema,
   hem `tail_vertex` hem `head_vertex` sütunlarında index içerir.)

3. **Farklı türdeki vertex'ler ve ilişkiler için farklı label'lar kullanarak** tek
   bir graph'ta birçok farklı bilgi türünü temiz bir data model'i koruyarak saklayabilirsiniz.

4. **Edge'ler tablosu**, **many-to-many associative/join table** gibi genelleştirilmiştir;
   tek bir tabloda birçok ilişki türü saklanabilir. Label ve property'lerde de index'ler
   olabilir.

> ⚠️ **Kısıtlama:** Bir edge yalnızca iki vertex'i birbirine bağlayabilir. Relational
> **join table** ise tek bir satırda birden fazla **foreign-key** referansı olduğunda
> üç yönlü veya daha yüksek dereceli ilişkileri temsil edebilir. Bu tür ilişkiler
> graph'ta, join table'ın her satırına karşılık gelen ek bir vertex ve o vertex'e/vertex'ten
> edge'ler oluşturularak ya da **hypergraph** kullanılarak temsil edilebilir.

### Graph model'in evolvability avantajı

Graph'lar **evolvability** (gelişebilirlik) açısından da avantajlıdır:
Uygulamanıza yeni özellikler eklendikçe, bir graph'ı uygulamanın veri yapılarındaki
değişikliklere uyum sağlamak için kolayca genişletebilirsiniz.

 ---
## 🔵 13. Cypher Query Language

### Cypher nedir?

**Cypher**, **property graph**'lar için bir sorgu dilidir. Başlangıçta **Neo4j** graph
veritabanı için oluşturulmuş, daha sonra **openCypher** olarak açık bir standarda
dönüştürülmüştür.

**Neo4j**'nin yanı sıra **Cypher**; **Memgraph, KùzuDB, Amazon Neptune, Apache AGE**
(PostgreSQL'de depolama) ve diğerleri tarafından desteklenmektedir.

> 📝 **Not:** Dil, The Matrix filmindeki bir karakterin adını taşır; kriptografideki
> "cipher" (şifre) ile ilgisi yoktur.

### Veri ekleme: CREATE

Graph'ın sol yarısını bir graph veritabanına eklemek için Cypher sorgusu:

```cypher
CREATE
  (namerica :Location {name:'North America', type:'continent'}),
  (usa      :Location {name:'United States', type:'country'  }),
  (idaho    :Location {name:'Idaho',         type:'state'    }),
  (lucy     :Person   {name:'Lucy' }),
  (idaho) -[:WITHIN ]-> (usa)  -[:WITHIN]-> (namerica),
  (lucy)  -[:BORN_IN]-> (idaho)
```

Her vertex'e sembolik bir isim verilir (`usa`, `idaho` gibi). Bu isimler veritabanında
saklanmaz; yalnızca sorgu içinde vertex'ler arasında edge oluşturmak için kullanılır.
Ok notasyonu kullanılır: `(idaho) -[:WITHIN]-> (usa)` ifadesi, `idaho` (**tail node**) ile
`usa` (**head node**) arasında `WITHIN` etiketli bir edge oluşturur.

### Veri sorgulama: MATCH

ABD'den Avrupa'ya göç etmiş kişilerin isimlerini bulmak için Cypher sorgusu:

```cypher
MATCH
  (person) -[:BORN_IN]->  () -[:WITHIN*0..]-> (:Location {name:'United States'}),
  (person) -[:LIVES_IN]-> () -[:WITHIN*0..]-> (:Location {name:'Europe'})
RETURN person.name
```

**Sorgunun okunuşu:**
Aşağıdaki iki koşulu sağlayan herhangi bir **vertex**'i (`person` olarak adlandırın) bulun:

1. `person`'ın giden bir `BORN_IN` edge'i var ve bu bir vertex'e işaret ediyor.
   O vertex'ten, sonunda adı `"United States"` olan bir `Location` türü vertex'e ulaşana
   dek giden `WITHIN` edge'lerini takip edebilirsiniz.

2. Aynı `person` vertex'inin giden bir `LIVES_IN` edge'i de var. Bu edge'i ve ardından
   giden `WITHIN` edge'lerini izleyerek sonunda adı `"Europe"` olan bir `Location`
   vertex'ine ulaşabilirsiniz.

3. Bu koşulları sağlayan her `person` vertex'i için `name` property'sini döndürün.

**`:WITHIN*0..`** ifadesi, "sıfır veya daha fazla kez WITHIN edge'ini takip et" anlamına gelir;
bir **regular expression**'daki `*` operatörüne benzer.

### Sorguyu çalıştırmanın farklı yolları

Cypher **declarative** olduğundan, sorguyu çalıştırmanın birden fazla yolu vardır:
- Tüm kişileri tarayıp her birinin doğum ve yaşadığı yeri inceleyebilirsiniz.
- Alternatif olarak, iki `Location` vertex'inden (**US** ve **Europe**) geriye doğru
  çalışabilirsiniz: İsim property'si için bir index varsa **US** ve **Europe**'u
  verimli şekilde bulabilir, ardından tüm gelen `WITHIN` edge'lerini izleyerek
  içlerindeki konumları bulabilir ve son olarak `BORN_IN` veya `LIVES_IN` edge'leri
  olan kişileri arayabilirsiniz.

**Query optimizer**, hangi stratejinin kullanılacağına kendisi karar verir.

---
## 🔗 14. Graph Queries in SQL (WITH RECURSIVE)

### Graph veri SQL ile sorgulanabilir mi?

Graph verisini relational yapıda saklamak mümkündür. Peki relational yapıya koyduktan
sonra SQL ile sorgulayabilir miyiz?

**Evet, ama zorluklarla birlikte.**

Bir graph sorgusunda geçtiğiniz her **edge**, esasen **edges** tablosuyla bir **join**'dir.
Relational veritabanında genellikle hangi **join**'lere ihtiyaç duyacağınızı önceden
bilirsiniz. Graph sorgusunda ise aradığınız **vertex**'i bulmadan önce değişken sayıda
**edge** geçmeniz gerekebilir — yani **join** sayısı önceden sabit değildir.

Cypher'daki `[:WITHIN*0..]` ifadesi bunu çok özlü biçimde ifade eder: bir kişi
`LIVES_IN` edge'i ile bir sokağa, şehre, ilçeye, bölgeye veya eyalete işaret edebilir;
ya da aradığınız konuma doğrudan işaret edebilir ya da birkaç seviye yukarıda olabilir.

### Recursive Common Table Expressions (WITH RECURSIVE)

SQL'de değişken uzunluklu geçiş yolları fikri **recursive common table expressions**
(**WITH RECURSIVE** sözdizimi) ile ifade edilebilir.

**Cypher'daki 4 satırlık sorgunun SQL'deki 31 satırlık karşılığı:**

```sql
WITH RECURSIVE
  -- in_usa: ABD içindeki tüm konumların vertex ID'leri
  in_usa(vertex_id) AS (
      SELECT vertex_id FROM vertices
        WHERE label = 'Location'
        AND properties->>'name' = 'United States'
    UNION
      SELECT edges.tail_vertex FROM edges
        JOIN in_usa ON edges.head_vertex = in_usa.vertex_id
        WHERE edges.label = 'within'
  ),

  -- in_europe: Avrupa içindeki tüm konumların vertex ID'leri
  in_europe(vertex_id) AS (
      SELECT vertex_id FROM vertices
        WHERE label = 'location'
        AND properties->>'name' = 'Europe'
    UNION
      SELECT edges.tail_vertex FROM edges
        JOIN in_europe ON edges.head_vertex = in_europe.vertex_id
        WHERE edges.label = 'within'
  ),

  -- born_in_usa: ABD'de doğan tüm kişilerin vertex ID'leri
  born_in_usa(vertex_id) AS (
    SELECT edges.tail_vertex FROM edges
      JOIN in_usa ON edges.head_vertex = in_usa.vertex_id
      WHERE edges.label = 'born_in'
  ),

  -- lives_in_europe: Avrupa'da yaşayan tüm kişilerin vertex ID'leri
  lives_in_europe(vertex_id) AS (
    SELECT edges.tail_vertex FROM edges
      JOIN in_europe ON edges.head_vertex = in_europe.vertex_id
      WHERE edges.label = 'lives_in'
  )

SELECT vertices.properties->>'name'
FROM vertices
JOIN born_in_usa     ON vertices.vertex_id = born_in_usa.vertex_id
JOIN lives_in_europe ON vertices.vertex_id = lives_in_europe.vertex_id;
```

**4 satırlık Cypher sorgusunun 31 satır SQL gerektirmesi**, doğru data model ve
sorgu dilinin seçiminin ne kadar büyük bir fark yaratabileceğini göstermektedir.

Bunun yanı sıra döngüleri işleme ve **breadth-first** veya **depth-first traversal**
arasında seçim yapma gibi detaylar da göz önünde bulundurulmalıdır.

**GQL (Graph Query Language) ISO standardı**, Cypher'a dayalı olup 2024'te yayımlandı.
Henüz geniş çapta benimsenmemiş olsa da önümüzdeki yıllarda graph veritabanları
arasında daha fazla tekdüzeliğe yol açması beklenmektedir.

---
## 🔺 15. Triple Stores ve SPARQL

### Triple store model nedir?

**Triple store** modeli, **property graph** modeline büyük ölçüde eşdeğerdir;
yalnızca aynı fikirleri farklı kelimelerle anlatır.

**Triple store**'da tüm bilgi çok basit üç parçalı ifadeler olarak saklanır:
**`(subject, predicate, object)`**

Örneğin `(Jim, likes, bananas)` triple'ında:
- `Jim` → **subject** (özne)
- `likes` → **predicate** (yüklem)
- `bananas` → **object** (nesne)

> 📝 **Teknik not:** **AWS Neptune** her triple'a bir graph ID ekleyerek **quads (4-tuple)**
> kullanır. **Datomic** her triple'ı bir transaction ID ve silme gösteren bir Boolean ile
> genişleterek **5-tuple** kullanır.

### Subject, predicate, object eşlemeleri

Bir triple'ın **subject**'i, graph'taki bir **vertex**'e eşdeğerdir.
**Object** ise iki şeyden biri olabilir:

1. **Primitive datatype değeri** (string veya number gibi):
   Bu durumda triple'ın **predicate** ve **object**'i, **subject vertex**'teki bir
   property'nin **key** ve **value**'suna eşdeğerdir.
   Örneğin `(lucy, birthYear, 1989)`, `{"birthYear": 1989}` properties'li bir
   `lucy` vertex'i gibidir.

2. **Başka bir vertex**:
   Bu durumda **predicate**, graph'taki bir **edge**'dir; **subject** **tail vertex**,
   **object** ise **head vertex**'tir.
   Örneğin `(lucy, marriedTo, alain)` triple'ında `lucy` ve `alain` her ikisi de
   **vertex**'tir; `marriedTo` ise bunları bağlayan **edge**'in label'ıdır.

### Turtle format (Notation3/N3 alt kümesi)

```turtle
@prefix : <urn:example:>.
_:lucy     a       :Person.
_:lucy     :name   "Lucy".
_:lucy     :bornIn _:idaho.
_:idaho    a       :Location.
_:idaho    :name   "Idaho".
_:idaho    :type   "state".
_:idaho    :within _:usa.
_:usa      a       :Location.
_:usa      :name   "United States".
_:usa      :type   "country".
_:usa      :within _:namerica.
_:namerica a       :Location.
_:namerica :name   "North America".
_:namerica :type   "continent".
```

Daha kompakt formu (noktalı virgülle aynı subject için birden fazla şey söylenebilir):

```turtle
@prefix : <urn:example:>.
_:lucy     a :Person;   :name "Lucy";          :bornIn _:idaho.
_:idaho    a :Location; :name "Idaho";         :type "state";   :within _:usa.
_:usa      a :Location; :name "United States"; :type "country"; :within _:namerica.
_:namerica a :Location; :name "North America"; :type "continent".
```

Bu örnekte graph'ın **vertex**'leri `_:someName` olarak yazılmıştır. Bu isim dosyanın
dışında bir anlam ifade etmez; yalnızca hangi triple'ların aynı vertex'e atıfta
bulunduğunu bilmek için gereklidir.

**Predicate bir edge'i temsil ettiğinde** object bir vertex'tir: `_:idaho :within _:usa`
**Predicate bir property olduğunda** object bir string literal'dir: `_:usa :name "United States"`

---
## 🌐 16. Semantic Web, RDF ve SPARQL

### Semantic Web nedir?

**Triple store** araştırma ve geliştirme çalışmalarının bir bölümü **Semantic Web**
tarafından motive edildi. 2000'lerin başındaki bu girişim, verileri yalnızca insan
tarafından okunabilir web sayfaları olarak değil, standardize edilmiş ve makine tarafından
okunabilir bir formatta da yayımlamak yoluyla internet genelinde veri alışverişini
kolaylaştırmayı amaçlıyordu.

**Semantic Web, başlangıçta tasarlandığı şekliyle başarılı olamadı.**
Yine de projenin mirası yaşamaya devam ediyor:

- **Linked data** standartları (**JSON-LD** gibi)
- Biyomedikal bilimdeki **ontologies**
- Facebook'un **Open Graph protocol**'ü (link unfurling için kullanılır)
- **Wikidata** gibi **knowledge graphs**
- **Schema.org** tarafından sağlanan yapılandırılmış veri standartları

**Triple stores**, orijinal kullanım alanı dışında da değerli uygulama görmüştür.
Semantic Web'e ilgi duymayan biri bile triple'ları uygulamalar için iyi bir
dahili data model olarak kullanabilir.

### RDF data model

**Turtle** dili aslında **RDF (Resource Description Framework)** veriyi kodlamanın
bir yoludur. **RDF**, **Semantic Web** için tasarlanmış bir data model'dir.
**Apache Jena** gibi araçlar, farklı **RDF encoding**'ler arasında otomatik dönüştürme yapabilir.

**RDF/XML** daha ayrıntılı bir kodlama biçimidir:

```xml
<rdf:RDF xmlns="urn:example:"
    xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
  <Location rdf:nodeID="idaho">
    <n>Idaho</n>
    <type>state</type>
    <within>
      <Location rdf:nodeID="usa">
        <n>United States</n>
        ...
      </Location>
    </within>
  </Location>
  <Person rdf:nodeID="lucy">
    <n>Lucy</n>
    <bornIn rdf:nodeID="idaho"/>
  </Person>
</rdf:RDF>
```

**RDF'nin bir tuhaflığı:** İnternet genelinde veri alışverişi için tasarlandığından,
triple'ların **subject, predicate** ve **object**'leri çoğunlukla **URI**'lardır.
Örneğin bir **predicate**, yalnızca `WITHIN` veya `LIVES_IN` değil,
`<http://my-company.com/namespace#within>` gibi bir URI olabilir.

Bu tasarımın arkasındaki gerekçe şudur: Kendi verinizi başkasının verisiyle
birleştirebilmeniz gerekir. Eğer farklı kişiler `within` kelimesine farklı anlamlar
atıyorsa, **predicate**'lerinin farklı **URI**'lar kullandığından çakışma olmaz.

### SPARQL query language

**SPARQL** (**S**PARQL **P**rotocol and **R**DF **Q**uery **L**anguage),
**RDF** data model kullanan **triple store**'lar için bir sorgu dilidir.
Cypher'dan önce gelir; Cypher'ın **pattern matching**'i **SPARQL**'dan ödünç alınmıştır;
bu nedenle birbirlerine çok benzerler.

**ABD'den Avrupa'ya taşınan kişileri bulan Cypher sorgusunun SPARQL karşılığı:**

```sparql
PREFIX : <urn:example:>

SELECT ?personName WHERE {
  ?person :name ?personName.
  ?person :bornIn  / :within* / :name "United States".
  ?person :livesIn / :within* / :name "Europe".
}
```

**Cypher ve SPARQL karşılaştırması:**

```
(person) -[:BORN_IN]-> () -[:WITHIN*0..]-> (location)   # Cypher
?person :bornIn / :within* ?location.                   # SPARQL
```

**RDF, property'ler ile edge'ler arasında ayrım yapmadığından** (her ikisi için de
**predicate** kullanır), property eşleştirmek için de aynı sözdizimini kullanabilirsiniz.

```
(usa {name:'United States'})   # Cypher
?usa :name "United States".    # SPARQL
```

**SPARQL** şu sistemler tarafından desteklenmektedir: Amazon Neptune, AllegroGraph,
Blazegraph, OpenLink Virtuoso, Apache Jena ve çeşitli diğer **triple store**'lar.

---
## 🧮 17. Datalog: Recursive Relational Queries

### Datalog nedir?

**Datalog**, **SPARQL** veya **Cypher**'dan çok daha eski bir dildir; 1980'lerdeki akademik
araştırmalardan doğmuştur. Yazılım mühendisleri arasında pek bilinmemekte ve ana akım
veritabanlarında geniş çapta desteklenmemektedir.

Ancak daha iyi bilinmeyi hak eder — karmaşık sorgular için özellikle güçlü,
son derece ifadeli bir dildir. **Datomic, LogicBlox, CozoDB** ve LinkedIn'in **LIquid**'i
dahil çeşitli niş veritabanları **Datalog**'u sorgu dili olarak kullanmaktadır.

**Datalog** bir **graph**'a değil, ilişkisel bir data model'e dayanır;
ancak özellikle **graph üzerindeki recursive queries** için güçlüdür.

**Datalog**, **Prolog** programlama dilinin bir alt kümesidir.

### Datalog facts

Bir **Datalog** veritabanının içeriği **facts** (olgular) olarak bilinir;
her **fact**, relational tablodaki bir satıra karşılık gelir.

Örneğin `location(2, "United States", "country")` fact'i,
`location` tablosunda ID'si 2 olan satırın bir ülke olduğunu belirtir.

```datalog
location(1, "North America", "continent").
location(2, "United States", "country").
location(3, "Idaho", "state").

within(2, 1).    /* US, North America içinde */
within(3, 2).    /* Idaho, US içinde         */

person(100, "Lucy").
born_in(100, 3). /* Lucy, Idaho'da doğmuş    */
```

### Datalog rules ve queries

Cypher ve SPARQL hemen `SELECT` ile başlar; **Datalog** ise adım adım ilerler.
Altta yatan **facts**'ten yeni **virtual tables** (sanal tablolar) türeten **rules**
(kurallar) tanımlarsınız.

```datalog
within_recursive(LocID, PlaceName) :- location(LocID, PlaceName, _). /* Kural 1 */

within_recursive(LocID, PlaceName) :- within(LocID, ViaID),          /* Kural 2 */
                                      within_recursive(ViaID, PlaceName).

migrated(PName, BornIn, LivingIn)  :- person(PersonID, PName),       /* Kural 3 */
                                      born_in(PersonID, BornID),
                                      within_recursive(BornID, BornIn),
                                      lives_in(PersonID, LivingID),
                                      within_recursive(LivingID, LivingIn).

us_to_europe(Person) :- migrated(Person, "United States", "Europe"). /* Kural 4 */
/* us_to_europe tablosu "Lucy" satırını içerir */
```

**Kuralların açıklaması:**

- **Kural 1:** Veritabanındaki her konum, kendisi de `within_recursive` içindedir.
- **Kural 2:** `within(LocID, ViaID)` ve `within_recursive(ViaID, PlaceName)` varsa,
  o zaman `within_recursive(LocID, PlaceName)` de geçerlidir. Bu kural **recursive**'dir;
  kendi kendini çağırır!
- **Kural 3:** `within_recursive` kullanarak belirli bir yerde doğan ve
  belirli bir yerde yaşayan kişileri bulur.
- **Kural 4:** ABD'den Avrupa'ya göç eden kişileri bulur.

### Datalog yaklaşımı

**Datalog** yaklaşımı, bu bölümde tartışılan diğer sorgu dillerinden farklı bir
düşünce biçimi gerektirir. Karmaşık sorguların kural kural oluşturulmasına olanak tanır;
bir kural diğer kurallara başvurabilir; bu, kodunuzu birbirini çağıran fonksiyonlara
bölmenize benzer. Fonksiyonlar **recursive** olabildiği gibi, **Datalog** kuralları da
kendini çağırabilir; bu da **Datalog** sorgularında **graph traversal**'ı mümkün kılar.

---
## 🔷 18. GraphQL

### GraphQL nedir?

**GraphQL**, bu bölümde gördüğümüz diğer sorgu dillerinden çok daha kısıtlayıcı
olacak şekilde tasarlanmış bir sorgu dilidir. **OLTP queries** için tasarlanmıştır;
amacı, kullanıcının cihazında çalışan **client software**'in (mobil uygulama veya
JavaScript web uygulaması gibi) UI'yi render etmek için gereken alanları içeren,
belirli bir yapıda bir **JSON document** talep etmesine olanak tanımaktır.

**GraphQL interfaces**, geliştiricilerin **server-side API**'leri değiştirmeden
**client code**'daki sorguları hızla değiştirebilmesine olanak tanır.

### GraphQL'in özellikleri ve kısıtlamaları

**GraphQL, kasıtlı olarak sınırlıdır:** Çünkü **GraphQL** sorguları güvenilmez
kaynaklardan gelir. Pahalıya mal olabilecek hiçbir şeye izin vermez; aksi takdirde
kullanıcılar çok sayıda pahalı sorgu çalıştırarak sunucuda **denial-of-service**
koşulu yaratabilir.

Özellikle:
- **Recursive queries**'e izin vermez (Cypher, SPARQL, SQL veya Datalog'un aksine)
- Servis sahipleri özellikle bu tür bir arama işlevi sunmayı seçmediği sürece
  **arbitrary search conditions**'a ("ABD'de doğup Avrupa'da yaşayan kişileri bul" gibi)
  izin vermez

**GraphQL esnekliğin bedeli:**
GraphQL benimseyen kuruluşların, sorguları genellikle **REST** veya **gRPC** kullanan
dahili servislere yönelik isteklere dönüştürmek için araçlara ihtiyacı vardır.
**Authorization, rate limiting** ve performans sorunları da ek endişelerdir.

### GraphQL örneği: Grup sohbet uygulaması

Discord veya Slack gibi bir grup sohbet uygulamasını **GraphQL** ile uygulamak:

```graphql
query ChatApp {
  channels {
    name
    recentMessages(latest: 50) {
      timestamp
      content
      sender {
        fullName
        imageUrl
      }
      replyTo {
        content
        sender {
          fullName
        }
      }
    }
  }
}
```

Bu sorgu, kullanıcının erişebildiği tüm **channel**'ları, her **channel**'ın adını
ve her **channel**'daki en son 50 mesajı talep eder. Her mesaj için:
- **timestamp** (zaman damgası)
- **content** (içerik)
- Gönderenin adı ve profil fotoğrafı **URL**'si
- Yanıt verilen mesaj varsa, o mesajın içeriği ve gönderenin adı

### Yanıt yapısı

```json
{
  "data": {
    "channels": [
      {
        "name": "#general",
        "recentMessages": [
          {
            "timestamp": 1693143014,
            "content": "Hey! How are y'all doing?",
            "sender": {"fullName": "Aaliyah", "imageUrl": "https://..."},
            "replyTo": null
          },
          {
            "timestamp": 1693143024,
            "content": "Great! And you?",
            "sender": {"fullName": "Caleb", "imageUrl": "https://..."},
            "replyTo": {
              "content": "Hey! How are y'all doing?",
              "sender": {"fullName": "Aaliyah"}
            }
          }
        ]
      }
    ]
  }
}
```

### GraphQL'in tasarım kararları

**Gönderenin bilgileri tekrarlanır:** Aynı kullanıcı birden fazla mesaj gönderirse,
bu bilgi her mesajda tekrar eder. Bu durum daha büyük bir yanıt boyutuna yol açar;
ama **GraphQL**, talep edilen veriye dayanarak UI'yi daha basit şekilde render etmek
için bu tasarım tercihini yapar.

**`replyTo` alanı da tekrar eder:** İkinci mesaj birinciye yanıttır; içerik ve gönderen
adı `replyTo` altında yinelenir. Bunun yerine yanıt verilen mesajın ID'si döndürülebilirdi;
ama bu, o ID son 50 mesaj arasında yoksa ek bir sunucu isteği yapılmasını gerektirirdi.
İçeriği kopyalamak, verilerle çalışmayı çok daha basit hale getirir.

**Önemli bir not:** **GraphQL** bir **response**'u bir **document database**'den gelen
yanıta benzese de ve adında "graph" geçse de, **GraphQL** herhangi bir veritabanı
türü (relational, document veya graph) üzerinde implement edilebilir.

---
## 📅 19. Event Sourcing ve CQRS

### Event Sourcing nedir?

Şimdiye kadar tartıştığımız tüm data modellerde veri, yazıldığı formu ile sorgulanır:
JSON documents, tablolardaki satırlar veya graph'taki vertex'ler ve edge'ler.

Ancak karmaşık uygulamalarda, verinin sorgulanması ve sunulması gereken tüm yolları
tatmin edebilecek tek bir veri temsili bulmak zorlaşabilir. Bu gibi durumlarda
veriyi bir formda yazıp, farklı okuma türleri için optimize edilmiş temsiller elde
etmek faydalı olabilir.

**Event Sourcing** bu ilkenin çok daha ileri götürülmesidir:
Bir olayı kaydetmek için her yazmada veriyi **self-contained** bir string olarak
kodlayıp (genellikle **JSON** olarak), bir zaman damgası ekleyip ardışık bir event
dizisine (**log**) eklersiniz.

**Event log'u**'ndaki olaylar **immutable**'dır (değişmezdir):
Hiçbir zaman değiştirilmez veya silinmez; yalnızca daha fazla event eklenir
(önceki event'lerin yerine geçebilir).

### Konferans yönetim sistemi örneği

Bir konferansın karmaşık bir iş alanı olduğunu düşünelim:
- Bireysel katılımcılar kayıt yaptırıp kredi kartıyla ödeme yapabilir.
- Şirketler toplu olarak yer satın alıp fatura ödeyebilir, ardından yerleri bireysel
  kişilere atayabilir.
- Konuşmacılar, sponsorlar ve gönüllüler için belirli sayıda yer ayrılabilir.
- Rezervasyonlar iptal edilebilir.
- Konferans organizatörü, etkinliği farklı bir salona taşıyarak kapasiteyi değiştirebilir.

Bu durumla birlikte, mevcut boş yer sayısını hesaplamak bile zorlu bir sorgu haline gelir.

**Event sourcing** yaklaşımında:
1. Konferansın her durum değişikliği önce bir **event** olarak saklanır
2. Her **event** log'a eklendiğinde, birkaç **materialized view** (veya **projection**,
   **read model**) güncellenerek o event'in etkisi yansıtılır:
   - Her rezervasyonun durumuna ilişkin bilgileri toplayan bir view
   - Konferans organizatörünün dashboard'u için grafikler hesaplayan başka bir view
   - Katılımcıların rozetlerini basan yazıcı için dosyalar oluşturan üçüncü bir view

### CQRS nedir?

**CQRS (Command Query Responsibility Segregation)**, ayrı **read-optimized representations**
sürdürme ve bunları **write-optimized representation**'dan türetme ilkesidir.

**Command** ve **fact** arasındaki fark:
- Kullanıcıdan bir istek geldiğinde bu bir **command** (komut) olarak adlandırılır;
  önce doğrulanması gerekir.
- Command yürütüldükten ve geçerli olduğu belirlendikten sonra (örneğin talep edilen
  rezervasyon için yeterli yer varsa), bir **fact** (olgu) haline gelir ve karşılık
  gelen **event** log'a eklenir.
- Event log yalnızca geçerli event'leri içermelidir.

**Event'leri geçmiş zamanda adlandırmak** önerilir (örneğin "seats were booked" yani
"yerler rezerve edildi"): Bir event, bir şeyin olduğunun kaydıdır.

### Event sourcing vs. Star schema fact table

**Event sourcing** ve **star schema fact table**'ı arasındaki benzerlik:
Her ikisi de geçmişte gerçekleşen event'lerin koleksiyonlarıdır.

Temel farklar:
- **Fact table**'daki satırların tümü aynı sütun kümesine sahipken,
  **event sourcing**'de farklı property'lere sahip birçok event türü olabilir.
- **Fact table** sırasız bir koleksiyonken, **event sourcing**'de event'lerin sırası önemlidir:
  Bir rezervasyon önce yapılıp sonra iptal edilirse, bu event'leri yanlış sırayla
  işlemek anlamsız olur.

### Event sourcing'in avantajları

1. **Niyet anlaşılırlığı:** Event'ler bir şeyin neden olduğunu daha iyi iletir.
   Örneğin "rezervasyon iptal edildi" event'ini anlamak,
   "rezervasyonlar tablosunun 4001. satırının `active` sütunu `false` yapıldı,
   `seat_assignments` tablosundan üç satır silindi ve `payments` tablosuna bir iade
   satırı eklendi" ifadesini anlamaktan çok daha kolaydır.

2. **Reproducibility (Yeniden üretilebilirlik):** **Materialized views** her zaman
   silinip event log'dan yeniden hesaplanabilir. **View maintenance** kodunda hata
   varsa, görünümü silip yeni kodla yeniden hesaplayabilirsiniz. Aynı kodu istediğiniz
   kadar yeniden çalıştırıp davranışını inceleyebilirsiniz.

3. **Birden fazla materialized view:** Uygulamanızın gerektirdiği belirli sorgular
   için optimize edilmiş birden fazla **materialized view** bulundurabilirsiniz.
   Bu view'lar event'lerle aynı ya da farklı bir veritabanında saklanabilir.
   Hızlı okuma için **denormalize** edilebilir; hatta servis her yeniden başladığında
   **event log**'dan yeniden hesaplandığı sürece yalnızca bellekte tutulabilir.

4. **Yeni view'lar oluşturmak kolaydır:** Mevcut **event log**'dan yeni bir
   **materialized view** oluşturmak kolaydır. Yeni event türleri veya mevcut
   event türlerine yeni property'ler ekleyerek yeni özellikleri desteklemek üzere
   sistemi geliştirebilirsiniz.

5. **Silme işlemini geri almak kolaylaşır:** Bir event hatayla yazıldıysa,
   onu tersine çeviren ardışık bir silme eventi yazabilirsiniz.
   **Downstream views** bu silme işlemini otomatik olarak dahil ederek veriyi düzeltir.

6. **Audit log:** Event log, sistemde neler olduğuna dair değerli bir **audit log**
   işlevi görür; bu, bu tür izlenebilirlik gerektiren sektörlerde önemlidir.

7. **Yüksek yazma throughput:** Event log'ları, **sequential access patterns**
   sayesinde veritabanlarından genellikle daha yüksek yazma throughput'u sağlar.
   Geçici event patlamaları tampon görevi gören log tarafından absorbe edilir.

### Event sourcing'in dezavantajları

1. **Dış bilgi sorunları:** Bir event döviz cinsinden bir fiyat içeriyorsa ve bir
   view için başka bir para birimine dönüştürülmesi gerekiyorsa, dış kaynaklı kur oranı
   her tarihte farklı olacağından tutarsızlık yaratır. Event işleme mantığını
   **deterministic** hale getirmek için kur oranını event'in kendisine dahil etmeniz
   ya da event'teki zaman damgasına göre tarihsel kuru sorgulamanız gerekir.

2. **Kişisel veri silme sorunu (GDPR):** Event'lerin **immutable** olması, kullanıcıların
   kişisel verilerinin silinmesini talep etmesini zorlaştırır (**GDPR** gibi yasal haklar).
   Event log kullanıcı bazlı ise, o kullanıcıya ait tüm log'u silebilirsiniz;
   ama log birden fazla kullanıcıyla ilgili event'leri içeriyorsa bu yöntem işe yaramaz.
   Kişisel veriyi event dışında saklamayı veya silebileceğiniz bir anahtarla şifrelemeyi
   deneyebilirsiniz (**crypto-shredding**); ama bu da türetilmiş durumu yeniden hesaplamayı
   zorlaştırır.

3. **Dış etkiler (side effects):** Event'lerin yeniden işlenmesi, **materialized view**'ı
   yeniden oluşturduğunuzda onay e-postalarını yeniden göndermek gibi istenmeyen dış
   etkilere yol açabilir.

### Event sourcing uygulayan sistemler

**Event sourcing**'i herhangi bir veritabanının üzerinde uygulayabilirsiniz; ama
bu pattern'i özellikle desteklemek için tasarlanmış sistemler de vardır:
**EventStoreDB, MartenDB** (PostgreSQL tabanlı), **Axon Framework**.

**Apache Kafka** gibi **message broker**'ları da event log'u saklamak için kullanabilirsiniz.
**Stream processor**'lar **materialized view**'ları güncel tutabilir.

Tek kritik gereksinim şudur: Event depolama sistemi, tüm **materialized view**'ların
event'leri log'da göründükleri sırayla işlemesini garanti etmelidir.

---
## 📊 20. DataFrames, Matrices ve Arrays

### Analitik ve bilimsel veri modelleri

Şimdiye kadar incelediğimiz data model'lerin tümü hem **transaction processing** hem de
**analytics** amaçları için kullanılmaktadır. Ancak analitik veya bilimsel bağlamda
sıklıkla karşılaşılan, **OLTP** sistemlerinde nadiren görülen birkaç data model daha vardır:
**DataFrames** ve sayı matrislerinden oluşan çok boyutlu diziler (**multidimensional arrays**).

### DataFrame nedir?

**DataFrame** data model'i, **R** dili, Python için **Pandas** kütüphanesi,
**Apache Spark, ArcticDB, Dask** ve diğer sistemler tarafından desteklenmektedir.

**DataFrame**'ler, **ML** modelleri için veri hazırlayan veri bilimciler arasında
popüler bir araçtır; ayrıca veri keşfi, istatistiksel veri analizi ve veri görselleştirme
gibi amaçlar için de yaygın biçimde kullanılır.

İlk bakışta **DataFrame**, relational database'deki bir tabloya veya bir **spreadsheet**'e
benzer. Bir **DataFrame** şu relational benzeri operatörleri destekler:

- Tüm satırlara bir fonksiyon uygulama
- Satırları bir koşula göre **filtering** (filtreleme)
- Satırları bazı sütunlara göre **grouping** ve diğer sütunları **aggregating** (toplayıp özetleme)
- İki DataFrame'i bir anahtara göre **joining** (birleştirme)

**SQL** gibi **declarative** bir sorgu dili kullanmak yerine, **DataFrame** genellikle
yapısını ve içeriğini değiştiren bir dizi komutla işlenir.

Bu yaklaşım, veri bilimcilerin tipik iş akışıyla uyumludur: Sordukları sorulara yanıt
bulmalarına olanak tanıyacak bir forma gelene kadar veriyi adım adım işlerler.
Bu işlemler genellikle veri bilimcinin kendi kopyası üzerinde gerçekleşir; bazen yerel
makinelerinde, ancak nihai sonuç diğer kullanıcılarla paylaşılabilir.

**DataFrame API'ları**, relational veritabanların sunduklarının çok ötesine geçen
işlemler sunar ve data model, tipik relational veri modellemesinden çok farklı
şekillerde kullanılır.

### DataFrame'den Matrix'e dönüşüm

**DataFrame**'lerin yaygın kullanımlarından biri, veriyi **relational-like**
bir temsilden **matrix** veya **multidimensional array** temsile dönüştürmektir;
bu da birçok **ML** algoritmasının girdiyi beklediği formdur.

**Örnek:** Film derecelendirmeleri relational → Matrix

```
Relational Tablo:
user_id | movie_id | rating
--------|----------|-------
1       | A        | 5
1       | C        | 3
2       | B        | 4
2       | C        | 2

Matrix Temsili (satır=kullanıcı, sütun=film):
      Film A  Film B  Film C
U1      5       -       3
U2      -       4       2
```

Matrix **sparse** (seyrek) olabilir — birçok kullanıcı-film kombinasyonu için veri yok;
ama bu sorun değil. Bu matrix'in binlerce sütunu olabilir ve relational veritabanına pek
uymaz; ama **DataFrame**'ler ve **sparse array**'ler sunan kütüphaneler (**NumPy** gibi)
bu tür veriyi kolayca işleyebilir.

### Sayısal olmayan veriyi matrix'e dönüştürmek

**Matrix yalnızca sayı içerebilir**, sayısal olmayan veriyi dönüştürmek için çeşitli
teknikler kullanılır:

- **Tarihler** → uygun bir aralıkta **floating-point numbers** olarak ölçeklendirme
- **Kategorik veriler (one-hot encoding):**
  Her olası değer için bir sütun oluşturulur ("komedi", "dram", "korku" gibi).
  Her film için, filmin kategorisine karşılık gelen sütuna `1`, diğer sütunlara `0` konur.
  Bu temsil kolayca birden fazla türe genelleşebilir.

### Matrix'in kullanım alanları

Veri sayılar matrisine dönüştürüldüğünde, birçok **ML** algoritmasının temelini
oluşturan **linear algebra** (doğrusal cebir) işlemlerine hazır hale gelir.

### Array databases ve time-series

**TileDB** gibi bazı veritabanları, büyük çok boyutlu sayı dizileri saklamada uzmanlaşmıştır;
bunlara **array databases** denir. En yaygın kullanım alanları:
- **Geospatial measurements** (düzenli aralıklı ızgarada raster veri)
- **Medical imaging** (tıbbi görüntüleme)
- Astronomik teleskoplardan gelen gözlemler

**DataFrame**'ler finans sektöründe de kullanılır:
Varlık fiyatları ve zaman içindeki işlemler gibi **time-series** verilerini temsil etmek için.

Popülerlikleri nedeniyle **DataFrame**'ler, **Apache Spark** ve **Flink** gibi **batch
processing frameworks**'e de eklenmiştir.

---
## 📋 21. Özet ve Karşılaştırma

### Ana veri modelleri

| Model | En İyi Olduğu Yer | Örnek Sistemler |
|-------|-------------------|----------------|
| **Relational Model** | Business analytics, OLTP, structured data | PostgreSQL, MySQL, Oracle |
| **Document Model** | Self-contained JSON docs, az ilişkiyle | MongoDB, Couchbase |
| **Graph Model** | Her şeyin her şeyle ilişkili olduğu durumlar | Neo4j, Amazon Neptune |
| **Event Sourcing** | Karmaşık iş alanları, audit trails | EventStoreDB, Kafka |
| **DataFrames/Arrays** | ML, veri analizi, bilimsel hesaplama | Pandas, Spark, NumPy |

### Sorgu dilleri karşılaştırması

| Dil | Tür | Hedef |
|-----|-----|-------|
| **SQL** | Declarative | Relational queries |
| **Cypher** | Declarative | Property graph queries |
| **SPARQL** | Declarative | Triple store / RDF queries |
| **Datalog** | Declarative + Recursive | Karmaşık graph traversals |
| **GraphQL** | Declarative (kısıtlı) | Client-server API queries |
| **MongoDB Aggregation** | Pipeline | JSON document analytics |

### Schema yaklaşımları

| Yaklaşım | Açıklama | Ne zaman? |
|----------|----------|-----------|
| **Schema-on-write** | Veri yazılırken doğrulanır (SQL) | Tüm kayıtlar aynı yapıya sahipse |
| **Schema-on-read** | Veri okunurken yorumlanır (JSON, MongoDB) | Heterojen, değişken yapıda veri |

### Normalization trade-off

```
NORMALIZED DATA
  + Yazması daha hızlı (tek kopya)
  + Tutarlılık kolaydır
  - Okuması yavaş (join gerekir)
  - Çok büyük sistemlerde join maliyeti sorun yaratır

DENORMALIZED DATA
  + Okuması hızlı (daha az join)
  - Yazması yavaş (çok kopya)
  - Tutarsızlık riski
  - Fazla disk alanı
```

### Event sourcing + CQRS pattern

```
Command gelir  →  Validate  →  Event log'a yazar
                                    ↓
               Materialized View 1 (dashboard)
               Materialized View 2 (badges printer)
               Materialized View 3 (booking status)
```

---
## 📚 22. Özel Veri Modelleri (Kısaca)

Bölüm, bazı veri modellerini kısaca bahsederek bitirir:

### Genome veritabanları
Genomik verilerle çalışan araştırmacılar genellikle **sequence similarity searches**
yapmaları gerekir: Bir DNA molekülünü temsil eden çok uzun bir string'i, benzer ama
aynı olmayan string'lerden oluşan büyük bir veritabanıyla eşleştirirler.
Bu kullanım durumunu hiçbir mevcut veritabanı karşılayamaz. Bu nedenle araştırmacılar
**GenBank** gibi özel genom veritabanı yazılımı geliştirmiştir.

### Muhasebe / Ledger sistemleri
Birçok finansal sistem, **double-entry accounting** (çift taraflı muhasebe) kullanan
**ledger**'ları veri modeli olarak kullanır.
Bu veriler relational veritabanlarında temsil edilebilir; ancak bu veri modeli için
özel veritabanlar da mevcuttur (**TigerBeetle** gibi).
**Cryptocurrencies** ve **blockchains**, genellikle veri modellerine yerleşik değer
transferi olan **distributed ledgers**'a dayalıdır.

### Full-text search
**Full-text search** (tam metin arama), veritabanlarıyla birlikte sıklıkla kullanılan
bir tür veri modeli olarak değerlendirilebilir.
**Information retrieval** geniş bir uzmanlık alanıdır. **Search indexes** ve **vector
search** konularına ilerideki bölümlerde değinilecektir.

---
## 📖 23. Anahtar Terimler Sözlüğü

| Terim | Türkçe Açıklama |
|-------|----------------|
| **Data model** | Verinin nasıl organize edildiğini ve işlendiğini tanımlayan yapı |
| **Abstraction** | Alt katmanın karmaşıklığını gizleyen soyutlama katmanı |
| **Declarative query** | Nasıl değil, ne istediğinizi belirttiğiniz sorgu dili |
| **Query optimizer** | Sorguyu en verimli şekilde çalıştırmayı planlayan veritabanı bileşeni |
| **Relational model** | Codd'un 1970'te önerdiği, tabloları/satırları kullanan veri modeli |
| **RDBMS** | Relational Database Management System |
| **NoSQL** | Yeni data model'ler, esneklik ve ölçeklenebilirlik odaklı gevşek fikir kümesi |
| **Document model** | JSON/XML belgeleri temel alan veri modeli |
| **Impedance mismatch** | OOP nesneleri ile relational tablolar arasındaki uyumsuzluk |
| **ORM** | Object-Relational Mapping — nesne ve tablo arasındaki çeviri katmanı |
| **N+1 query problem** | Her ilişkili kayıt için ayrı sorgu atan ORM anti-pattern'i |
| **One-to-many** | Bir kaydın çok sayıda ilişkili kaydı olduğu ilişki türü |
| **Many-to-many** | Her iki tarafın da çok sayıda ilişkili kaydı olduğu ilişki türü |
| **Normalization** | Verinin yalnızca bir yerde saklandığı (ID referans) tasarım yaklaşımı |
| **Denormalization** | Verinin birden fazla yerde kopyalandığı (hızlı okuma için) yaklaşım |
| **Join** | İlişkili tablolardaki verileri birleştiren SQL işlemi |
| **Hydrating IDs** | ID'leri insan tarafından okunabilir bilgiye dönüştürme (uygulama kodu join'i) |
| **Schema-on-read** | Veri okunurken yorumlanan örtük şema (document databases) |
| **Schema-on-write** | Veri yazılırken doğrulanan açık şema (relational databases) |
| **Data locality** | İlgili verinin aynı anda okunması için yakın saklanması |
| **Shredding** | Document-like yapıyı ayrı tablolara bölmek |
| **Star schema** | Ortada fact table, etrafında dimension tables bulunan analytics schema'sı |
| **Snowflake schema** | Dimension'ların subdimension'lara bölündüğü, daha normalize star schema |
| **Fact table** | Her satırın bir event'i temsil ettiği analytics merkezi tablo |
| **Dimension table** | Fact'lerin detaylarını içeren (kim, ne, nerede...) analytics tablosu |
| **OBT** | One Big Table — dimension'ları fact table'a katan denormalize yaklaşım |
| **Graph** | Vertex'ler (düğümler) ve edge'ler (kenarlar) ile oluşan ağ yapısı |
| **Vertex** | Graph'taki düğüm/node |
| **Edge** | Graph'taki iki vertex arasındaki bağlantı |
| **Property graph** | Her vertex ve edge'in label ve properties taşıdığı graph modeli |
| **Cypher** | Neo4j için geliştirilmiş, openCypher standardındaki graph sorgu dili |
| **Triple store** | (subject, predicate, object) üçlüleri saklayan veri deposu |
| **RDF** | Resource Description Framework — Semantic Web için veri modeli |
| **SPARQL** | RDF triple store'lar için sorgu dili |
| **Turtle** | RDF verisini yazmak için okunabilir bir format (Notation3 alt kümesi) |
| **Semantic Web** | 2000'lerde internet genelinde makine-okunabilir veri alışverişi girişimi |
| **Knowledge graph** | Varlıklar ve aralarındaki ilişkiler hakkında bilgi saklayan graph |
| **Datalog** | 1980'lerden Prolog tabanlı, recursive sorgular için güçlü sorgu dili |
| **GraphQL** | Client-server API için kısıtlı graph sorgu dili |
| **GQL** | Graph Query Language — Cypher tabanlı ISO standardı (2024) |
| **Event sourcing** | Tüm değişikliklerin immutable event log'a yazıldığı veri tasarım pattern'i |
| **CQRS** | Command Query Responsibility Segregation — read ve write modellerini ayırma |
| **Command** | Kullanıcıdan gelen, doğrulanması gereken istek |
| **Event** | Gerçekleşmiş bir olayın kaydı (immutable, geçmiş zaman) |
| **Materialized view / Projection** | Event'lerden türetilen, okuması optimize edilmiş veri temsili |
| **Crypto-shredding** | Kişisel veriyi şifreleme anahtarını silerek geri dönüşümsüz silme tekniği |
| **DataFrame** | Pandas/Spark'ta kullanılan tablo-benzeri, bulk işlemlere uygun veri yapısı |
| **Sparse matrix** | Çoğu değerin 0 veya boş olduğu seyrek matris |
| **One-hot encoding** | Kategorik değerleri 0/1 sütunlarına dönüştürme tekniği |
| **Array database** | Çok boyutlu sayı dizilerini saklayan özel veritabanı (TileDB gibi) |
| **Adjacency list** | Her vertex'in komşularını sakladığı graph temsil yöntemi |
| **Adjacency matrix** | Graph'ı vertex x vertex matris olarak temsil etme yöntemi |
| **WITH RECURSIVE** | SQL'de değişken derinlikteki graph traversal'lara izin veren sözdizimi |
| **PageRank** | Web graph üzerinde sayfa popülerliğini ölçen Google algoritması |